# QuoteGridBuilder — Chunked Ingestion Demo

Demonstrates building a `QuoteGrid` from multiple DataFrame chunks,
then feeding it directly into the online solver.

In [ ]:
import math
import time

import polars as pl
import price_contour as pc

print(f"price_contour {pc.__version__}")

## 1. Generate synthetic data

Small dataset: 500 quotes × 11 multiplier steps.

In [ ]:
def make_df(n_quotes=500, n_steps=11, seed=42):
    """Generate synthetic scored DataFrame."""
    mults = [0.80 + 0.04 * j for j in range(n_steps)]
    rows = []
    for q in range(n_quotes):
        elasticity = 1.5 + 3.5 * q / n_quotes
        base = 80.0 + 40.0 * q / n_quotes
        for j, mult in enumerate(mults):
            conversion = 1.0 / (1.0 + math.exp(elasticity * (mult - 1.0)))
            rows.append({
                "quote_id": f"Q{q:05d}",
                "scenario_step": j,
                "multiplier": mult,
                "expected_income": base * mult * conversion,
                "volume": conversion,
                "loss_ratio": 0.6 / mult * (1.0 + 0.1 * (mult - 1.0)),
            })
    return pl.DataFrame(
        rows,
        schema={
            "quote_id": pl.Utf8,
            "scenario_step": pl.Int32,
            "multiplier": pl.Float32,
            "expected_income": pl.Float32,
            "volume": pl.Float32,
            "loss_ratio": pl.Float32,
        },
    )


df = make_df()
print(f"Shape: {df.shape}")
df.head()

## 2. One-shot solve (baseline)

Pass the full DataFrame directly to `OnlineOptimiser.solve()`. This is the existing workflow.

In [ ]:
solver = pc.OnlineOptimiser(
    objective="expected_income",
    constraints={"volume": {"min": 0.90}},
    max_iter=100,
)

t0 = time.perf_counter()
result_df = solver.solve(df)
elapsed_df = time.perf_counter() - t0

print(f"DataFrame path: {elapsed_df:.3f}s")
print(f"  Objective:  {result_df.total_objective:,.2f}")
print(f"  Converged:  {result_df.converged}")
print(f"  Iterations: {result_df.iterations}")
print(f"  Lambdas:    {result_df.lambdas}")

## 3. Chunked build via QuoteGridBuilder

Split data into 5 chunks, feed them through the builder, then solve from the grid.

In [ ]:
# Split into 5 chunks (by quote, not by row)
n_quotes = 500
n_steps = 11
chunk_quotes = 100

builder = pc.QuoteGridBuilder(
    ["volume"],  # constraint columns
    objective="expected_income",
)

for start in range(0, n_quotes, chunk_quotes):
    end = start + chunk_quotes
    chunk = df.filter(
        pl.col("quote_id").is_in([f"Q{q:05d}" for q in range(start, end)])
    )
    builder.append(chunk)
    print(f"  Appended chunk {start//chunk_quotes + 1}: {chunk.shape[0]} rows")

grid = builder.build()
print(f"\nGrid: {grid}")
print(f"  n_quotes: {grid.n_quotes}")
print(f"  n_steps:  {grid.n_steps}")
print(f"  multipliers: {grid.multipliers[:5]}... ({len(grid.multipliers)} total)")
print(f"  constraints: {grid.constraint_names}")

In [ ]:
# Solve directly from the grid
t0 = time.perf_counter()
result_grid = solver.solve(grid)
elapsed_grid = time.perf_counter() - t0

print(f"Grid path: {elapsed_grid:.3f}s")
print(f"  Objective:  {result_grid.total_objective:,.2f}")
print(f"  Converged:  {result_grid.converged}")
print(f"  Iterations: {result_grid.iterations}")
print(f"  Lambdas:    {result_grid.lambdas}")

## 4. Verify: DataFrame path == Grid path

In [ ]:
obj_diff = abs(result_df.total_objective - result_grid.total_objective)
print(f"Objective difference: {obj_diff:.6f}")
assert obj_diff < 1e-2, f"Mismatch: {obj_diff}"
print("Results match.")

## 5. Error handling

The builder raises clear errors for misuse.

In [ ]:
# Append after build raises
try:
    builder.append(df)
    print("ERROR: should have raised")
except ValueError as e:
    print(f"Append after build: {e}")

# Empty builder raises on build
try:
    empty = pc.QuoteGridBuilder(["volume"], objective="expected_income")
    empty.build()
    print("ERROR: should have raised")
except ValueError as e:
    print(f"Empty build: {e}")